<a href="https://colab.research.google.com/github/KalinaMarkova/deep_learning_course_project/blob/main/04_Model_Training_Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Grained Analysis of Propaganda in News Articles
## Notebook 04: Hierarchical Two-Stage Pipeline: Binary Span Detector & Technique Classifier - Experiment 2

In this notebook we will implement a Hierarchical Two-Model Architecture to detect propaganda in news articles. Instead of forcing a single model to solve boundary extraction and 14-class technique classification simultaneously, we divide the task into two specialized stages:

**Model A (Binary Span Detector)**: A token-classification model trained strictly on 3 labels (O, B-PROPAGANDA, I-PROPAGANDA). Its sole purpose is to draw precise bounding boxes around manipulative language spans.

**Model B (Technique Classifier)**: A sequence-classification model that takes the isolated text spans identified by Model A and categorizes them into their specific propaganda classes (e.g., Loaded Language, Slogans, Name Calling).

In [ ]:
!pip install -q transformers datasets seqeval evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import numpy as np
import torch
import torch.nn as nn

Let's load the dataset and split into training, validation and test datasets.

In [ ]:
drive.mount('/content/drive')

# 1. Load the flat dataset from your correct Drive path
dataset_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/exp2_span_3labels_sentence_dataset'
dataset = load_from_disk(dataset_path)

# 2. Perform the Three-Way Split (80% Train, 10% Val, 10% Test)
# First, separate 80% for training and 20% for the temporary hold-out
train_temp_split = dataset.train_test_split(test_size=0.20, seed=42)
train_dataset = train_temp_split['train']
temp_dataset = train_temp_split['test']

# Next, split that 20% hold-out evenly into 10% Validation and 10% Test
val_test_split = temp_dataset.train_test_split(test_size=0.50, seed=42)
val_dataset = val_test_split['train']
test_dataset = val_test_split['test']

# 3. Recreate the 3-class Label Dictionaries perfectly
exp2_labels_list = ['O', 'B-PROPAGANDA', 'I-PROPAGANDA']

label2id = {label: i for i, label in enumerate(exp2_labels_list)}
id2label = {i: label for label, i in label2id.items()}

# Load RoBERTa Tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

print(f"Dataset split is completed.")
print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

Mounted at /content/drive


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Dataset split is completed.
Train size: 12022
Validation size: 1503
Test size: 1503


Let's prepare the evaluation function so the Trainer can track Precision, Recall, and F1 score at the end of each epoch.

In [ ]:
# Load seqeval metric
metric = evaluate.load("seqeval")

def compute_metrics(p):
    """Calculates Precision, Recall, F1, and Accuracy ignoring -100 padding tokens."""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Filter out -100 (special/padding tokens)
    true_predictions = [
        [exp2_labels_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [exp2_labels_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Let's initilaize **Model A** and start training.

In [ ]:
# 1. Initialize the model
model_a = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. Add attention masks (RoBERTa needs to know which tokens are real and which are padding)
def add_attention_mask(example):
    return {"attention_mask": [1] * len(example["input_ids"])}

if "attention_mask" not in train_dataset.column_names:
    print("Adding attention masks to datasets...")
    train_dataset = train_dataset.map(add_attention_mask)
    val_dataset = val_dataset.map(add_attention_mask)
    test_dataset = test_dataset.map(add_attention_mask)

# 3. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./model_a_span_detector",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# 4. Initialize Data Collator (dynamically pads sentences to the same length)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 5. Initialize the Trainer
trainer_a = Trainer(
    model=model_a,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


# Start training
trainer_a.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Adding attention masks to datasets...


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.273639,0.267262,0.089820,0.104712,0.096696,0.899468
2,0.238127,0.272499,0.112299,0.109948,0.111111,0.902283
3,0.129818,0.314276,0.130904,0.169284,0.147641,0.897383


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2256, training_loss=0.22212588528157973, metrics={'train_runtime': 655.1687, 'train_samples_per_second': 55.048, 'train_steps_per_second': 3.443, 'total_flos': 1618540840813428.0, 'train_loss': 0.22212588528157973, 'epoch': 3.0})

Although Accuracy is steady at ~90%, the F1 Score peaks at only 13.8% (0.1378). This happens because ~90% of all tokens in standard news text are non-propaganda (O). If a model predicts O for every single word in the dataset, it achieves 90% accuracy with an F1 score of 0. In Epoch 3, training Loss plummeted to 0.131, but Validation Loss jumped up to 0.314. This indicates that the model began memorizing the training data rather than generalizing.

In order to recify the low F1 score, we will introducing a penalty multiplier that makes missing a rare propaganda word cost the model 8 times more than missing a regular word, forcing it to actively hunt for manipulation rather than taking the easy route of ignoring it. In addition, we will reduce the number of training epochs to avoid overfitting.

In [ ]:
# 1. Define Class Weights (Pushing the model to hunt for 1 and 2)
# Move weights to the same device as the model (GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = torch.tensor([1.0, 8.0, 8.0]).to(device)

# 2. Create the Custom Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Apply the weights to the CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)

        # Flatten predictions and labels for the loss function
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# 3. Update Training Arguments (Reduced to 2 epochs to prevent overfitting)
weighted_training_args = TrainingArguments(
    output_dir="./model_a_span_detector_weighted",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,              # <--- Reduced to 2 epochs
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# 4. Initialize the Weighted Trainer
weighted_trainer_a = WeightedTrainer(
    model=model_a,
    args=weighted_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,      # <--- Using the correct updated argument
    data_collator=data_collator,
    compute_metrics=compute_metrics
)



# 5. Train
weighted_trainer_a.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.226655,0.947860,0.076976,0.195462,0.110454,0.863768
2,0.188894,1.188140,0.095275,0.214660,0.131974,0.872631


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1504, training_loss=0.22806295641559235, metrics={'train_runtime': 453.546, 'train_samples_per_second': 53.013, 'train_steps_per_second': 3.316, 'total_flos': 1079655297747840.0, 'train_loss': 0.22806295641559235, 'epoch': 2.0})

The class weights worked and Recall almost doubled from 11.5% (in the previous Epoch 2) to 20.9%. This means the model found nearly twice as much actual propaganda text as it did before. As a result, F1 Score also improved at 13.4% compared to 11.6% previously.

However, because we are punishing the model for missed propaganda 8x more than a regular mistake, it started guessing B-PROPAGANDA and I-PROPAGANDA much more aggressively. Precision dropped slightly to ~9.8%.

The Validationloss jumped to over 1.15. This happens with weighted loss functions because when the model is wrong on an 8x-weighted token, the mathematical penalty is massive, inflating the loss number even if the core metrics (F1/Recall) are improving.



Let's initialize a fresh RoBERTa model, with the following adjustments:

1. Lowering the multiplier to 4.0 maintains a strong incentive to find rare propaganda tokens without destroying model precision.
2. Lowering the learning rate (`learning_rate=2e-5`) prevents the model from making drastic weight updates when it encounters complex, fuzzy span boundaries, helping the loss converge smoothly.
3. Restoring Epochs to 3, with gentler learning rates and softened weights, the model will learn at a controlled pace, allowing it to benefit from 3 full epochs without instantly overfitting on Epoch 3 like it did previously.
4. Add a `warmup_ratio=0.1` to force the learning rate to start at 0 and slowly climb to 2e-5 over the first 10% of the data. At the start of fine-tuning, the randomly initialized classification head produces giant, unstable gradients. Warming up the learning rate gradually over the first 10% of training protects the pre-trained RoBERTa weights from being ruined early on.



In [9]:
# 1. Start with a fresh model
model_a_improved = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. Softened Class Weights (Dialed back from 8.0 to 4.0)
device = "cuda" if torch.cuda.is_available() else "cpu"
soft_weights = torch.tensor([1.0, 4.0, 4.0]).to(device)

# 3. Create the Custom Trainer with Soft Weights
class ImprovedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Apply the softened weights
        loss_fct = nn.CrossEntropyLoss(weight=soft_weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# 4. Optimized Training Arguments
improved_training_args = TrainingArguments(
    output_dir="./model_a_span_detector_optimized",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# 5. Initialize the Improved Trainer
improved_trainer_a = ImprovedTrainer(
    model=model_a_improved,
    args=improved_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


improved_trainer_a.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.500453,0.482616,0.045407,0.150087,0.069720,0.845584
2,0.430761,0.494835,0.075703,0.183246,0.107143,0.868585
3,0.272189,0.569074,0.090708,0.214660,0.127527,0.873089


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2256, training_loss=0.43032868090250814, metrics={'train_runtime': 759.0029, 'train_samples_per_second': 47.518, 'train_steps_per_second': 2.972, 'total_flos': 1618540840813428.0, 'train_loss': 0.43032868090250814, 'epoch': 3.0})

 Precision, Recall, and F1 score improved steadily across all three epochs, peaking at Epoch 3 (F1: 12.75%, Recall: 21.47%, Precision: 9.07%).

 The Validation Loss remained much lower and more stable (0.48 to 0.57) than the 8x run (1.18), confirming that softening the weights prevented extreme loss spikes.

 Because the penalty was cut in half, the model learned more conservatively — it took 3 full epochs to reach the exact same Recall level (21.47%) that the 8x model achieved in just 2 epochs.

 At Epoch 3, the Training Loss dropped sharply ($0.43 \rightarrow 0.27$) while the Validation Loss began creeping back up ($0.49 \rightarrow 0.57$), indicating that 3 epochs is the absolute limit before overfitting begins.

# Evaluation

For **Model A** we would prefer the 8x Weighted Model as it gaves us higher F1 and Precision in fewer epochs. Let's evalaute the model using the competition partial metrics.

In [10]:
# 1. Get raw predictions from the 8x Weighted Model on the Test Set
print("Extracting test predictions from Model A (8x Weighted Trainer)...")
raw_preds, raw_labels, _ = weighted_trainer_a.predict(test_dataset)
pred_ids = np.argmax(raw_preds, axis=2)

# 2. Format prediction and label IDs into lists of BIO tag strings (ignoring -100 padding)
true_references = [
    [exp2_labels_list[l] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(pred_ids, raw_labels)
]

pred_references = [
    [exp2_labels_list[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(pred_ids, raw_labels)
]

# --- Span Extraction Function ---
def extract_spans(tags):
    """Converts a list of BIO tags into spans: (label, start_index, end_index)"""
    spans = []
    current_span = None

    for i, tag in enumerate(tags):
        if tag == 'O':
            if current_span:
                spans.append(current_span)
                current_span = None
        elif tag.startswith('B-'):
            if current_span:
                spans.append(current_span)
            current_span = (tag[2:], i, i)
        elif tag.startswith('I-'):
            if current_span and current_span[0] == tag[2:]:
                # Extend current span
                current_span = (current_span[0], current_span[1], i)
            else:
                # Malformed I-tag (starts without a B-tag)
                if current_span:
                    spans.append(current_span)
                current_span = (tag[2:], i, i)

    if current_span:
        spans.append(current_span)
    return spans

# --- Partial Overlap Score Calculation ---
total_true_spans = 0
total_pred_spans = 0
total_partial_recall_score = 0.0
total_partial_precision_score = 0.0

# Evaluate sentence by sentence
for true_tags, pred_tags in zip(true_references, pred_references):
    true_spans = extract_spans(true_tags)
    pred_spans = extract_spans(pred_tags)

    total_true_spans += len(true_spans)
    total_pred_spans += len(pred_spans)

    # 1. Calculate Partial Recall
    for t_label, t_start, t_end in true_spans:
        t_length = t_end - t_start + 1
        best_overlap = 0

        for p_label, p_start, p_end in pred_spans:
            if t_label == p_label:
                overlap_start = max(t_start, p_start)
                overlap_end = min(t_end, p_end)
                if overlap_start <= overlap_end:
                    overlap_len = overlap_end - overlap_start + 1
                    best_overlap = max(best_overlap, overlap_len)

        total_partial_recall_score += (best_overlap / t_length)

    # 2. Calculate Partial Precision
    for p_label, p_start, p_end in pred_spans:
        p_length = p_end - p_start + 1
        best_overlap = 0

        for t_label, t_start, t_end in true_spans:
            if p_label == t_label:
                overlap_start = max(p_start, t_start)
                overlap_end = min(p_end, t_end)
                if overlap_start <= overlap_end:
                    overlap_len = overlap_end - overlap_start + 1
                    best_overlap = max(best_overlap, overlap_len)

        total_partial_precision_score += (best_overlap / p_length)

# Calculate final percentages
partial_precision = total_partial_precision_score / total_pred_spans if total_pred_spans > 0 else 0
partial_recall = total_partial_recall_score / total_true_spans if total_true_spans > 0 else 0

if (partial_precision + partial_recall) > 0:
    partial_f1 = 2 * (partial_precision * partial_recall) / (partial_precision + partial_recall)
else:
    partial_f1 = 0.0

print("=" * 60)
print(" SEMEVAL-STYLE PARTIAL OVERLAP SCORES (8x Weighted Model)")
print("=" * 60)
print(f"Total Gold Spans (Ground Truth) : {total_true_spans}")
print(f"Total Predicted Spans (Model)   : {total_pred_spans}")
print("-" * 60)
print(f"Partial Precision               : {partial_precision:.4f} ({partial_precision*100:.1f}%)")
print(f"Partial Recall                  : {partial_recall:.4f} ({partial_recall*100:.1f}%)")
print(f"Partial F1 Score                : {partial_f1:.4f} ({partial_f1*100:.1f}%)")
print("=" * 60)

Extracting test predictions from Model A (8x Weighted Trainer)...


 SEMEVAL-STYLE PARTIAL OVERLAP SCORES (8x Weighted Model)
Total Gold Spans (Ground Truth) : 563
Total Predicted Spans (Model)   : 1259
------------------------------------------------------------
Partial Precision               : 0.3411 (34.1%)
Partial Recall                  : 0.6131 (61.3%)
Partial F1 Score                : 0.4384 (43.8%)


Achieving 61.3% Partial Recall on fuzzy, highly subjective text boundaries is a good result for a base RoBERTa model. It means the model is successfully capturing over 60% of the actual manipulative text in the dataset.

However, the model is predicting more than twice as many spans as actually exist to avoid the high penalty. This explains the lower Precision (34.1%). It is flagging a lot of borderline or innocent text just to be safe.

For **Model A**, we want the model to over-predict rather than under-predict. If it misses a span entirely, that text is lost forever. If it accidentally flags an innocent sentence, Model B might learn to recognize it as a false positive.

Let's save the selected model.

In [11]:
# 1. Define the target directory in your Drive
drive_save_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_a_span_detector_8x'

# 2. Save the model and tokenizer
weighted_trainer_a.save_model(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_a_span_detector_8x/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_a_span_detector_8x/tokenizer.json')